# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs. All references are made using their `@id`.

In [ ]:
# List all available record sets and fields by their @id
print("Available record sets and their @id:")
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets found in the metadata. If the dataset schema is valid Croissant, record_sets should be defined.")
else:
    for rs in record_sets:
        print(f"- RecordSet @id: {rs['@id']} | name: {rs.get('name', 'n/a')}")
        if 'field' in rs and rs['field']:
            for field in rs['field']:
                print(f"  - Field @id: {field['@id']} | name: {field.get('name', 'n/a')}")
        print()


## 3. Data Extraction
Load data from each available record set into a DataFrame for analysis. Use the record set and field `@id`s as referenced in the overview above.

In [ ]:
# Extract records from each record set using its @id
record_set_ids = [rs['@id'] for rs in list(dataset.record_sets)]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Show available DataFrames with their columns
if not dataframes:
    print("No records or record sets were found. Please verify the Croissant schema and dataset content.")
else:
    # Select the first record set as default for demonstration
    first_record_set_id = list(dataframes.keys())[0]
    print(f"Columns in record set {first_record_set_id}:")
    print(dataframes[first_record_set_id].columns.tolist())
    dataframes[first_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data across different record sets. **All fields should be referenced using their `@id`.**

In [ ]:
# For EDA, use the first available DataFrame and select numeric and group fields by their @id
import numpy as np

if not dataframes:
    print("No data to analyze.")
else:
    df = dataframes[first_record_set_id]
    
    # Attempt to infer a numeric field and a categorical field by data type
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dtype, np.number):
            numeric_field_id = col
            break
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field_id:
            group_field_id = col
            break
    if not numeric_field_id:
        print("No numeric field found; cannot proceed with numeric EDA.")
    else:
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / (filtered_df[numeric_field_id].std() + 1e-8)
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # If group field exists, group by it and compute mean
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No suitable group/categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field and its relation to the group field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print("No data to plot.")
elif not numeric_field_id:
    print("No numeric field for visualization.")
else:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library, with all entities referenced strictly by their `@id`. We examined metadata, loaded records from each record set, performed basic EDA, and visualized variable distributions. Further, domain-specific analysis can be carried out leveraging the structured and interoperable schema provided by Croissant.